# Soil Recommendation — Data Pipeline V4
### V2 (6 Global Datasets) + V3 (Official MALR Egypt) + Smart Cleaning
---
| Source | Type | Approx. Size |
|--------|-------|----------------|
| Kaggle DS1 — Crop Recommendation | Global | ~2,200 |
| Kaggle DS2 — Fertilizer Prediction | Global | ~1,500 |
| Kaggle DS3 — Fertilizer Recommendation | Global | ~2,000 |
| Kaggle DS4 — Crop Recommender Soil Nutrients | Global | ~2,000 |
| Mendeley DS5 — Crop + Soil | Global | Large |
| FAO DS6 — Egypt | Egyptian Statistical | ~1,549 |
| **MALR Egypt Synthetic** | **Official Egyptian 🇪🇬** | **~7,000** |

### Cleaning Strategy
Instead of dropping every row with NULL, we apply **Tiered Cleaning**:
- **Tier 1 (Core):** rows with N+P+K+crop → for classification
- **Tier 2 (Full):** rows with N+P+K+crop+pH → for full recommendation
- **Tier 3 (Egypt):** complete MALR rows → for Egyptian context

##  STEP 1 — Install Libraries

In [ ]:
!pip install kaggle datasets -q
import pandas as pd
import numpy as np
import json, os, random, glob
from google.colab import files
print(' Done')

## 🇪🇬 STEP 2 — Official MALR Egypt Data
> Source: FAO/MALR 2024 — Official Egyptian Ministry of Agriculture recommendations

In [ ]:
# ══════════════════════════════════════════════════
# Ideal values — Source: MALR Egypt 2024
# Unit: N,P,K in kg/ha
# ══════════════════════════════════════════════════
EGYPT_MALR_IDEAL = {
    # Main crops
    'cotton':       {'N':157,  'P':55,  'K':60,  'pH':7.2, 'soil':'clay loam'},
    'faba bean':    {'N':40,   'P':70,  'K':60,  'pH':7.0, 'soil':'clay'},
    'maize':        {'N':252,  'P':55,  'K':0,   'pH':6.8, 'soil':'clay loam'},
    'potato':       {'N':300,  'P':145, 'K':115, 'pH':6.0, 'soil':'sandy loam'},
    'rice':         {'N':120,  'P':40,  'K':0,   'pH':6.5, 'soil':'clay'},
    'sugarcane':    {'N':380,  'P':75,  'K':115, 'pH':6.5, 'soil':'clay loam'},
    'tomato':       {'N':300,  'P':110, 'K':115, 'pH':6.5, 'soil':'sandy loam'},
    'wheat':        {'N':170,  'P':40,  'K':0,   'pH':7.0, 'soil':'clay loam'},
    # Field crops and legumes
    'barley':       {'N':45,   'P':15,  'K':60,  'pH':7.0, 'soil':'sandy loam'},
    'broad bean':   {'N':15,   'P':70,  'K':60,  'pH':7.0, 'soil':'clay'},
    'flax':         {'N':45,   'P':40,  'K':0,   'pH':6.5, 'soil':'clay loam'},
    'lentil':       {'N':15,   'P':30,  'K':60,  'pH':6.5, 'soil':'clay loam'},
    'sugar beet':   {'N':60,   'P':80,  'K':0,   'pH':7.0, 'soil':'clay loam'},
    'green peas':   {'N':15,   'P':70,  'K':60,  'pH':6.5, 'soil':'sandy loam'},
    # Aromatic plants
    'aniseed':      {'N':145,  'P':70,  'K':60,  'pH':6.5, 'soil':'sandy loam'},
    'coriander':    {'N':145,  'P':70,  'K':60,  'pH':6.5, 'soil':'sandy loam'},
    'mint':         {'N':430,  'P':110, 'K':60,  'pH':6.5, 'soil':'loam'},
    # Fodder crops
    'clover':       {'N':90,   'P':62,  'K':57,  'pH':6.5, 'soil':'clay loam'},
    'sorghum':      {'N':310,  'P':70,  'K':0,   'pH':6.8, 'soil':'clay loam'},
    # Fruits
    'mango':        {'N':215,  'P':70,  'K':87,  'pH':6.5, 'soil':'sandy loam'},
    'citrus':       {'N':310,  'P':70,  'K':78,  'pH':6.5, 'soil':'sandy loam'},
    'grapes':       {'N':197,  'P':92,  'K':85,  'pH':6.5, 'soil':'sandy loam'},
    'banana':       {'N':1070, 'P':215, 'K':115, 'pH':6.5, 'soil':'loam'},
    'apple':        {'N':215,  'P':70,  'K':87,  'pH':6.5, 'soil':'loam'},
    # Vegetables
    'onion':        {'N':257,  'P':75,  'K':87,  'pH':6.5, 'soil':'sandy loam'},
    'garlic':       {'N':162,  'P':75,  'K':87,  'pH':6.5, 'soil':'sandy loam'},
    'eggplant':     {'N':250,  'P':55,  'K':60,  'pH':6.5, 'soil':'loam'},
    'pepper':       {'N':250,  'P':55,  'K':60,  'pH':6.5, 'soil':'loam'},
    'cabbage':      {'N':130,  'P':55,  'K':60,  'pH':6.8, 'soil':'clay loam'},
    'squash':       {'N':130,  'P':55,  'K':60,  'pH':6.5, 'soil':'loam'},
    'beans':        {'N':130,  'P':70,  'K':115, 'pH':6.5, 'soil':'loam'},
    'strawberry':   {'N':550,  'P':110, 'K':230, 'pH':6.0, 'soil':'sandy loam'},
    'artichoke':    {'N':145,  'P':65,  'K':60,  'pH':6.5, 'soil':'clay loam'},
    'sweet potato': {'N':70,   'P':55,  'K':60,  'pH':6.0, 'soil':'sandy loam'},
}

# Mapping Kaggle names → MALR
CROP_MAP = {
    'maize':'maize','corn':'maize','rice':'rice','wheat':'wheat',
    'cotton':'cotton','potato':'potato','tomato':'tomato',
    'sugarcane':'sugarcane','lentil':'lentil','mango':'mango',
    'grapes':'grapes','banana':'banana','apple':'apple',
    'orange':'citrus','citrus':'citrus','onion':'onion',
    'garlic':'garlic','barley':'barley','sorghum':'sorghum',
    'chickpea':'green peas','kidneybeans':'beans','motherbeans':'beans',
    'mungbean':'faba bean','blackgram':'faba bean','pigeonpeas':'faba bean',
    'groundnut':'beans','watermelon':'squash','muskmelon':'squash',
    'papaya':'mango','pomegranate':'citrus','coconut':'mango',
    'coffee':'mango','jute':'flax','soybean':'beans','sunflower':'flax',
}
DEFAULT_IDEAL = {'N':120,'P':60,'K':60,'pH':7.0,'soil':'clay loam'}

def get_ideal(crop):
    c = str(crop).lower().strip()
    return EGYPT_MALR_IDEAL.get(CROP_MAP.get(c,c), DEFAULT_IDEAL)

print(f'MALR Egypt: {len(EGYPT_MALR_IDEAL)} crops loaded')

##  STEP 3 — Generate Egyptian Synthetic Data

In [ ]:
EGYPT_FERTS = [
    'urea','calcium nitrate','ammonium sulphate',
    'triple superphosphate','single superphosphate',
    'potassium sulphate','potassium chloride',
    'NPK 20-20-0','NPK 15-15-15','DAP'
]
EGYPT_SOILS = ['clay','clay loam','sandy loam','sandy','loam','calcareous']

def gen_egypt_data(n_per_crop=250, noise=0.15):
    rows = []
    for crop, ideal in EGYPT_MALR_IDEAL.items():
        N_id,P_id,K_id,pH_id = ideal['N'],ideal['P'],ideal['K'],ideal['pH']
        for _ in range(n_per_crop):
            fac = lambda: max(0, 1 + np.random.uniform(-noise*2, noise*1.5))
            rows.append({
                'nitrogen':   round(N_id  * fac(), 1) if N_id  > 0 else 0,
                'phosphorus': round(P_id  * fac(), 1) if P_id  > 0 else 0,
                'potassium':  round(K_id  * fac(), 1) if K_id  > 0 else 0,
                'ph':         round(min(9.5, max(5.0, pH_id + np.random.uniform(-0.8,0.8))), 2),
                'crop':       crop,
                'fertilizer': random.choice(EGYPT_FERTS),
                'soil_type':  ideal.get('soil', random.choice(EGYPT_SOILS)),
                'temperature':round(np.random.uniform(15,38), 1),
                'humidity':   round(np.random.uniform(30,75), 1),
                'rainfall':   round(np.random.uniform(5,50),  1),
                'moisture':   round(np.random.uniform(20,60), 1),
                'source':     'MALR-Egypt',
            })
    return pd.DataFrame(rows)

df_egypt = gen_egypt_data(n_per_crop=250)
print(f'Egypt synthetic: {len(df_egypt)} rows | {df_egypt["crop"].nunique()} crops')

##  STEP 4 — Kaggle API

In [ ]:
files.upload()  # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print('Kaggle ready')

##  STEP 5 — Download 4 Datasets from Kaggle

In [ ]:
os.makedirs('./data', exist_ok=True)
for slug, folder in [
    ('atharvaingle/crop-recommendation-dataset',                     'ds1'),
    ('gdabhishek/fertilizer-prediction',                             'ds2'),
    ('ismailsaleem/fertilizer-recommendation-dataset',               'ds3'),
    ('manikantasanjayv/crop-recommender-dataset-with-soil-nutrients', 'ds4'),
]:
    os.makedirs(f'./data/{folder}', exist_ok=True)
    os.system(f'kaggle datasets download -d {slug} -p ./data/{folder} --unzip -q')
    print(f'{folder}: {[os.path.basename(c) for c in glob.glob(f"./data/{folder}/*.csv")]}')

##  STEP 6 — Upload DS5 (Mendeley) + DS6 (FAO)

In [ ]:
print('Upload DS5 from: data.mendeley.com/datasets/vynxnppr7j/1')
u5 = files.upload()
os.makedirs('./data/ds5', exist_ok=True)
for fn,d in u5.items(): open(f'./data/ds5/{fn}','wb').write(d)
print(f'DS5: {list(u5.keys())}')

In [ ]:
print('Upload FAO CSV file:')
u6 = files.upload()
os.makedirs('./data/ds6', exist_ok=True)
for fn,d in u6.items(): open(f'./data/ds6/{fn}','wb').write(d)
print(f'DS6: {list(u6.keys())}')

## STEP 7 — Preview Columns
>  Check columns and adjust col_map in STEP 8 if needed

In [ ]:
for ds in ['ds1','ds2','ds3','ds4','ds5','ds6']:
    csvs = glob.glob(f'./data/{ds}/*.csv')
    if not csvs: print(f'{ds}: '); continue
    df_t = pd.read_csv(csvs[0], nrows=2)
    full = pd.read_csv(csvs[0])
    print(f'\n{ds} | shape={full.shape} | cols={df_t.columns.tolist()}')
    print(f'  nulls: {full.isnull().sum().to_dict()}')

## STEP 8 — Load and Standardize 6 Datasets + MALR

In [ ]:
STD = ['nitrogen','phosphorus','potassium','ph','crop','fertilizer',
       'temperature','humidity','rainfall','moisture','soil_type','source']

def load_std(path, col_map, src):
    try:
        df = pd.read_csv(path).rename(columns=col_map)
        for c in STD:
            if c not in df.columns: df[c] = np.nan
        df['source'] = src
        print(f'   {src}: {len(df)} rows')
        return df[STD]
    except Exception as e:
        print(f'   {src}: {e}'); return pd.DataFrame()

print('Loading datasets...')

df1 = load_std(glob.glob('./data/ds1/*.csv')[0], {
    'N':'nitrogen','P':'phosphorus','K':'potassium','ph':'ph',
    'label':'crop','temperature':'temperature',
    'humidity':'humidity','rainfall':'rainfall'
}, 'Kaggle-CropRec')

df2 = load_std(glob.glob('./data/ds2/*.csv')[0], {
    'Nitrogen':'nitrogen','Phosphorous':'phosphorus','Potassium':'potassium',
    'Crop Type':'crop','Fertilizer Name':'fertilizer',
    'Temparature':'temperature','Humidity ':'humidity','Humidity':'humidity',
    'Moisture':'moisture','Soil Type':'soil_type'
}, 'Kaggle-FertPred')

df3 = load_std(glob.glob('./data/ds3/*.csv')[0], {
    'Nitrogen':'nitrogen','Phosphorus':'phosphorus','Potassium':'potassium',
    'Crop Type':'crop','Fertilizer Name':'fertilizer',
    'Soil Type':'soil_type','Humidity':'humidity',
    'Temperature':'temperature','Moisture':'moisture'
}, 'Kaggle-FertRec')

df4_list = [load_std(p, {
    'N':'nitrogen','P':'phosphorus','K':'potassium','ph':'ph',
    'label':'crop','temperature':'temperature',
    'humidity':'humidity','rainfall':'rainfall'
}, 'Kaggle-SoilNut') for p in glob.glob('./data/ds4/*.csv')]
df4 = pd.concat(df4_list, ignore_index=True) if df4_list else pd.DataFrame()

ds5_csv = glob.glob('./data/ds5/*.csv')
df5 = load_std(ds5_csv[0], {
    'N':'nitrogen','P':'phosphorus','K':'potassium',
    'ph':'ph','pH':'ph','label':'crop','crop':'crop','Crop':'crop',
    'temperature':'temperature','Temperature':'temperature',
    'humidity':'humidity','Humidity':'humidity',
    'rainfall':'rainfall','Rainfall':'rainfall',
    'Soil Type':'soil_type','Fertilizer':'fertilizer'
}, 'Mendeley') if ds5_csv else pd.DataFrame()

ds6_csv = glob.glob('./data/ds6/*.csv')
if ds6_csv:
    fao = pd.read_csv(ds6_csv[0])
    fao_use = fao[fao['Element']=='Agricultural Use'].copy()
    df6 = pd.DataFrame({'fertilizer': fao_use['Item'].str.lower().str.strip(), 'source':'FAO-Egypt'})
    for c in STD:
        if c not in df6.columns: df6[c] = np.nan
    df6 = df6[STD]
    print(f'  FAO-Egypt: {len(df6)} rows')
else:
    df6 = pd.DataFrame()

# MALR Egypt
df_egypt_std = df_egypt.copy()
for c in STD:
    if c not in df_egypt_std.columns: df_egypt_std[c] = np.nan
df_egypt_std = df_egypt_std[STD]
print(f'  MALR-Egypt: {len(df_egypt_std)} rows')

## STEP 9 — Merge All Sources
### Then Tiered Cleaning Instead of Random Dropping

In [ ]:
all_dfs = [df for df in [df1,df2,df3,df4,df5,df6,df_egypt_std] if not df.empty]
df_all  = pd.concat(all_dfs, ignore_index=True)

print(f' Before Cleaning:')
print(f'  Total rows : {len(df_all)}')
print(f'  Sources    : {df_all["source"].value_counts().to_dict()}')
print(f'\n  NULL per column:')
null_pct = (df_all.isnull().sum() / len(df_all) * 100).round(1)
for col, pct in null_pct.items():
    bar = '█' * int(pct/5) + '░' * (20-int(pct/5))
    print(f'  {col:<15} {bar} {pct}%')

In [ ]:
# ══════════════════════════════════════════════════════
# TIERED CLEANING — preserving maximum data possible
# ══════════════════════════════════════════════════════
df_c = df_all.copy()

# 1. Normalize text first
for col in ['crop','fertilizer','soil_type']:
    df_c[col] = df_c[col].astype(str).str.lower().str.strip()
    df_c[col] = df_c[col].replace({'nan':np.nan,'none':np.nan,'':np.nan,'0':np.nan})

# 2. Drop rows without N, P, or K (core columns)
before = len(df_c)
df_c = df_c.dropna(subset=['nitrogen','phosphorus','potassium'])
print(f'After dropping without NPK    : {len(df_c)} (removed {before-len(df_c)})')

# 3. Drop rows without crop
before = len(df_c)
df_c = df_c.dropna(subset=['crop'])
print(f'After dropping without crop   : {len(df_c)} (removed {before-len(df_c)})')

# 4. Remove NPK outliers
before = len(df_c)
for col in ['nitrogen','phosphorus','potassium']:
    q1,q99 = df_c[col].quantile(0.005), df_c[col].quantile(0.995)
    df_c = df_c[(df_c[col]>=q1) & (df_c[col]<=q99)]
print(f'After removing outliers       : {len(df_c)} (removed {before-len(df_c)})')

# 5. pH cleaning — not dropping rows, filling with ideal values
def fill_ph(row):
    if pd.notna(row['ph']) and 3 <= row['ph'] <= 10:
        return row['ph']
    ideal = get_ideal(row['crop'])
    return ideal['pH']  # fill with ideal value instead of dropping

df_c['ph'] = df_c.apply(fill_ph, axis=1)
print(f'After pH handling (fill)      : {len(df_c)} (no drops — filled with ideal)')

# 6. soil_type — not dropping, filling from MALR
def fill_soil(row):
    if pd.notna(row['soil_type']): return row['soil_type']
    return get_ideal(row['crop']).get('soil', 'clay loam')
df_c['soil_type'] = df_c.apply(fill_soil, axis=1)

# 7. Drop duplicates
before = len(df_c)
df_c = df_c.drop_duplicates(subset=['nitrogen','phosphorus','potassium','crop'])
print(f'After dropping duplicates     : {len(df_c)} (removed {before-len(df_c)})')

print(f'\n Final result: {len(df_c)} rows')
print(f'Sources: {df_c["source"].value_counts().to_dict()}')
print(f'Crops  : {df_c["crop"].nunique()} unique')

In [ ]:
# Split into Tiers to know how many rows are ready per recommendation type
tier1 = df_c.dropna(subset=['nitrogen','phosphorus','potassium','crop'])
tier2 = df_c.dropna(subset=['nitrogen','phosphorus','potassium','crop','ph'])
tier3 = df_c[df_c['source']=='MALR-Egypt']

print(f' Tier Split:')
print(f'  Tier 1 (NPK+crop)      : {len(tier1):,} rows  → classification')
print(f'  Tier 2 (NPK+crop+pH)   : {len(tier2):,} rows  → full recommendation')
print(f'  Tier 3 (Egypt MALR)    : {len(tier3):,} rows  → Egyptian context')

##  STEP 10 — Data Augmentation (3x data expansion)

In [ ]:
def augment(df, n=2, noise=0.05):
    num_cols = [c for c in ['nitrogen','phosphorus','potassium','ph',
                             'temperature','humidity','rainfall','moisture']
                if c in df.columns]
    parts = [df.copy()]
    for _ in range(n):
        aug = df.copy()
        for col in num_cols:
            m = aug[col].notna()
            aug.loc[m,col] = (
                aug.loc[m,col] +
                np.random.normal(0,noise,m.sum()) * aug.loc[m,col]
            ).clip(0).round(2)
        parts.append(aug)
    return pd.concat(parts, ignore_index=True).drop_duplicates(
        subset=['nitrogen','phosphorus','potassium','crop']
    )

df_aug = augment(df_c, n=2, noise=0.05)
print(f' After Augmentation : {len(df_aug):,} rows')
print(f'   Egypt rows      : {len(df_aug[df_aug["source"]=="MALR-Egypt"]):,}')
print(f'   Global rows     : {len(df_aug[df_aug["source"]!="MALR-Egypt"]):,}')

## STEP 11 — Hybrid Recommendation Sheet Function

In [ ]:
FERT_REC = {
    'N_low':   'Urea (46-0-0) or Calcium Nitrate (15.5-0-0)',
    'N_high':  'Nitrogen surplus — stop adding nitrogen fertilizer',
    'P_low':   'Triple Superphosphate (0-46-0) or Single Superphosphate (0-18-0)',
    'P_high':  'Phosphorus surplus — do not add phosphate fertilizer',
    'K_low':   'Potassium Sulphate SOP (0-0-50) or MOP (0-0-60)',
    'K_high':  'Potassium surplus — do not add potassium fertilizer',
    'pH_low':  'Add agricultural lime (Calcium Carbonate) to raise pH',
    'pH_high': 'Add agricultural sulfur to lower pH',
}
THR = 0.12

def hybrid_report(row):
    crop  = str(row.get('crop','unknown') or 'unknown').strip().lower()
    N_cur = round(float(row.get('nitrogen',0)   or 0), 1)
    P_cur = round(float(row.get('phosphorus',0) or 0), 1)
    K_cur = round(float(row.get('potassium',0)  or 0), 1)
    pH_cur= round(float(row.get('ph', 7.0) or 7.0), 2)
    fert  = str(row.get('fertilizer','') or '')
    soil  = str(row.get('soil_type','') or '')
    src   = str(row.get('source','') or '')

    ideal = get_ideal(crop)
    N_id,P_id,K_id,pH_id = ideal['N'],ideal['P'],ideal['K'],ideal['pH']
    N_gap = round(N_id - N_cur, 1)
    P_gap = round(P_id - P_cur, 1)
    K_gap = round(K_id - K_cur, 1)
    pH_gap= round(pH_id - pH_cur, 2)

    def sym(gap, id_val, unit='kg/ha'):
        thr = max(id_val * THR, 3)
        if abs(gap) <= thr:  return '✅', 'Ideal'
        elif gap > 0:        return '🔴', f'Deficient by {abs(gap)} {unit}'
        else:                return '🟡', f'Excess by {abs(gap)} {unit}'

    Ns,Nst   = sym(N_gap,  N_id)
    Ps,Pst   = sym(P_gap,  P_id)
    Ks,Kst   = sym(K_gap,  K_id)
    pHs,pHst = sym(pH_gap, pH_id, unit='')

    ref = 'MALR Egypt 🇪🇬' if 'Egypt' in src or 'MALR' in src else 'Global reference'
    soil_str = soil.title() if soil and soil not in ['nan','none',''] else ideal.get('soil','clay loam').title()

    L = [
        '📋 Soil Improvement Recommendation Sheet',
        f'Crop: {crop.title()}  |  Soil Type: {soil_str}  |  Reference: {ref}',
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━',
        f'{"Element":<20}{"Current":>12}{"Ideal":>12}   Status',
        '──────────────────────────────────────────────',
        f'{"Nitrogen (N)":<20}{str(N_cur)+" kg/ha":>12}{str(N_id)+" kg/ha":>12}   {Ns} {Nst}',
        f'{"Phosphorus (P)":<20}{str(P_cur)+" kg/ha":>12}{str(P_id)+" kg/ha":>12}   {Ps} {Pst}',
        f'{"Potassium (K)":<20}{str(K_cur)+" kg/ha":>12}{str(K_id)+" kg/ha":>12}   {Ks} {Kst}',
        f'{"Soil pH":<20}{str(pH_cur):>12}{str(pH_id):>12}   {pHs} {pHst}',
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━',
        '🧪 Recommendations:'
    ]

    has_rec = False
    for key, gap, id_val, label in [
        ('N', N_gap, N_id, 'Nitrogen'),
        ('P', P_gap, P_id, 'Phosphorus'),
        ('K', K_gap, K_id, 'Potassium'),
    ]:
        if abs(gap) <= max(id_val*THR, 3): continue
        d = 'low' if gap > 0 else 'high'
        if gap > 0:
            L.append(f'• {label}: Add {abs(gap)} kg/ha')
            L.append(f'  → {FERT_REC[key+"_"+d]}')
        else:
            L.append(f'• {label}: {FERT_REC[key+"_"+d]}')
        has_rec = True

    if abs(pH_gap) > 0.4:
        d = 'low' if pH_gap > 0 else 'high'
        L.append(f'• pH: Current={pH_cur} → Ideal={pH_id}')
        L.append(f'  → {FERT_REC["pH_"+d]}')
        has_rec = True

    if fert and fert not in ['nan','none','']:
        L.append(f'• Recommended compound fertilizer: {fert.title()}')
        has_rec = True

    if not has_rec:
        L.append('•  Soil is in ideal condition — no adjustments needed')

    L += [
        '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━',
        f'Note: Ideal values according to {ref} for {crop.title()} crop'
    ]
    return '\n'.join(L)

# Test
print(hybrid_report({
    'crop':'wheat','nitrogen':100,'phosphorus':30,'potassium':0,
    'ph':7.8,'fertilizer':'urea','soil_type':'clay loam','source':'MALR-Egypt'
}))

##  STEP 12 — Generate Instruction + Conversational Pairs

In [ ]:
INST_T = [
    "I have agricultural soil and want to grow {crop}.\nAnalysis results:\n- N: {N} kg/ha\n- P: {P} kg/ha\n- K: {K} kg/ha\n- pH: {pH}\n- Soil type: {soil}\nCan you prepare a full recommendation sheet?",
    "A farmer has a soil analysis for {crop} crop:\nN={N} | P={P} | K={K} | pH={pH} | Soil: {soil}\nNeeds fertilizer recommendations according to MALR.",
    "Soil report — Crop: {crop}\nValues: N={N}, P={P}, K={K}, pH={pH}, Soil type={soil}\nWhat are the required recommendations?",
    "I have a soil analysis: N:{N} P:{P} K:{K} pH:{pH} ({soil})\nTarget crop: {crop}\nGive me the full recommendation sheet.",
]
Q_T = [
    "What is the best fertilizer for {crop}? N={N}, P={P}, K={K}, pH={pH}",
    "My soil: N={N} P={P} K={K} pH={pH}. I want to grow {crop}. What should I do?",
    "How do I improve my soil for {crop}? N={N}, P={P}, K={K}, pH={pH}",
    "Is this soil suitable for {crop}? N:{N} P:{P} K:{K} pH:{pH} — what is the recommendation?",
    "What fertilizers are needed for {crop} with N={N} P={P} K={K} pH={pH}?",
]

inst_pairs, conv_pairs = [], []

for _, row in df_aug.iterrows():
    try:
        crop = str(row.get('crop','?') or '?').strip()
        N    = round(float(row.get('nitrogen',0)   or 0), 1)
        P    = round(float(row.get('phosphorus',0) or 0), 1)
        K    = round(float(row.get('potassium',0)  or 0), 1)
        pH   = round(float(row.get('ph',7.0) or 7.0), 2)
        soil = str(row.get('soil_type','clay loam') or 'clay loam')
        soil = soil if soil not in ['nan','none',''] else 'clay loam'

        out  = hybrid_report(row)
        if len(out) < 100: continue

        inst = random.choice(INST_T).format(
            crop=crop.title(), N=N, P=P, K=K, pH=pH, soil=soil)
        inst_pairs.append({'instruction':inst,'input':'','output':out})
    except: pass

for _, row in df_aug.sample(min(4000,len(df_aug)), random_state=42).iterrows():
    try:
        crop = str(row.get('crop','?') or '?').strip()
        N    = round(float(row.get('nitrogen',0)   or 0), 1)
        P    = round(float(row.get('phosphorus',0) or 0), 1)
        K    = round(float(row.get('potassium',0)  or 0), 1)
        pH   = round(float(row.get('ph',7.0) or 7.0), 2)
        q    = random.choice(Q_T).format(crop=crop.title(),N=N,P=P,K=K,pH=pH)
        out  = hybrid_report(row)
        conv_pairs.append({'instruction':q,'input':'','output':out})
    except: pass

print(f' Instruction pairs : {len(inst_pairs):,}')
print(f' Conversational    : {len(conv_pairs):,}')
print(f' Total             : {len(inst_pairs)+len(conv_pairs):,}')

##  STEP 13 — Save Final Data

In [ ]:
all_pairs = inst_pairs + conv_pairs
random.shuffle(all_pairs)
total = len(all_pairs)
tr, vl = int(total*.85), int(total*.92)

os.makedirs('./output', exist_ok=True)

def save_jsonl(data, path):
    with open(path,'w',encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False)+'\n')

save_jsonl(all_pairs[:tr],   './output/train.jsonl')
save_jsonl(all_pairs[tr:vl], './output/val.jsonl')
save_jsonl(all_pairs[vl:],   './output/test.jsonl')
df_aug.to_csv('./output/merged_dataset.csv', index=False)
with open('./output/egypt_malr_ideal.json','w',encoding='utf-8') as f:
    json.dump(EGYPT_MALR_IDEAL, f, ensure_ascii=False, indent=2)

print(f' Files saved:')
print(f'  Train : {tr:,} pairs')
print(f'  Val   : {vl-tr:,} pairs')
print(f'  Test  : {total-vl:,} pairs')
for fn in os.listdir('./output'):
    print(f'  {fn}: {os.path.getsize(f"./output/{fn}")//1024} KB')

## STEP 14 — Quality Check

In [ ]:
with open('./output/train.jsonl','r',encoding='utf-8') as f:
    samples = [json.loads(l) for l in f.readlines()[:3]]

for i,s in enumerate(samples):
    print(f'\n{"="*55}\n[{i+1}] INSTRUCTION:\n{s["instruction"]}')
    print(f'\n[{i+1}] OUTPUT:\n{s["output"]}')

print(f'\n Statistics:')
print(f'  Avg instruction len : {np.mean([len(p["instruction"]) for p in all_pairs]):.0f} chars')
print(f'  Avg output len      : {np.mean([len(p["output"]) for p in all_pairs]):.0f} chars')
print(f'\n🎉 Data is ready for Fine-tuning on Mistral-7B + QLoRA!')

##  STEP 14.5 — Fill Remaining NULLs Without Dropping Any Row
### Columns: fertilizer + rainfall + moisture

In [ ]:
# ══════════════════════════════════════════════════════════════
# Default fertilizer lookup per crop (from MALR + agricultural research)
# ══════════════════════════════════════════════════════════════
DEFAULT_FERTILIZER = {
    'wheat':        'urea',
    'maize':        'urea',
    'rice':         'ammonium sulphate',
    'cotton':       'calcium nitrate',
    'potato':       'NPK 15-15-15',
    'tomato':       'NPK 20-20-0',
    'sugarcane':    'urea',
    'barley':       'urea',
    'faba bean':    'single superphosphate',
    'lentil':       'single superphosphate',
    'sugar beet':   'DAP',
    'onion':        'NPK 15-15-15',
    'garlic':       'NPK 15-15-15',
    'eggplant':     'NPK 20-20-0',
    'pepper':       'NPK 20-20-0',
    'cabbage':      'urea',
    'squash':       'NPK 20-20-0',
    'beans':        'DAP',
    'strawberry':   'NPK 15-15-15',
    'mango':        'NPK 20-20-0',
    'citrus':       'NPK 15-15-15',
    'grapes':       'potassium sulphate',
    'banana':       'potassium sulphate',
    'apple':        'NPK 15-15-15',
    'sorghum':      'urea',
    'clover':       'single superphosphate',
    'groundnut':    'DAP',
    'soybean':      'DAP',
    'sunflower':    'NPK 20-20-0',
    'chickpea':     'single superphosphate',
    'watermelon':   'NPK 20-20-0',
    'muskmelon':    'NPK 20-20-0',
    'papaya':       'NPK 20-20-0',
    'coconut':      'potassium sulphate',
    'coffee':       'NPK 15-15-15',
    'jute':         'urea',
    'blackgram':    'DAP',
    'mungbean':     'DAP',
    'pigeonpeas':   'DAP',
    'kidneybeans':  'DAP',
    'motherbeans':  'DAP',
    'pomegranate':  'NPK 15-15-15',
}
DEFAULT_FERT_FALLBACK = 'NPK 15-15-15'

# ══════════════════════════════════════════════════════════════
# Default rainfall lookup per crop (mm/year)
# Source: FAO Crop Water Requirements + MALR
# ══════════════════════════════════════════════════════════════
DEFAULT_RAINFALL = {
    'wheat':        25.0,   # Egypt — mostly irrigated farming
    'maize':        20.0,
    'rice':         20.0,   # Irrigated
    'cotton':       15.0,
    'potato':       20.0,
    'tomato':       15.0,
    'sugarcane':    20.0,
    'barley':       30.0,
    'faba bean':    25.0,
    'lentil':       25.0,
    'sugar beet':   20.0,
    'onion':        15.0,
    'garlic':       15.0,
    'eggplant':     15.0,
    'pepper':       15.0,
    'mango':        20.0,
    'citrus':       20.0,
    'grapes':       25.0,
    'banana':       20.0,
    'sorghum':      20.0,
    'clover':       25.0,
    'groundnut':    30.0,
    'soybean':      30.0,
    'sunflower':    25.0,
    'chickpea':     40.0,
    'watermelon':   20.0,
    'coconut':      25.0,
    'coffee':       40.0,
    'jute':         60.0,
}
DEFAULT_RAIN_FALLBACK = 20.0  # Egypt average

# ══════════════════════════════════════════════════════════════
# Default moisture lookup per crop (%)
# ══════════════════════════════════════════════════════════════
DEFAULT_MOISTURE = {
    'rice':         70.0,   # Requires very moist soil
    'sugarcane':    65.0,
    'banana':       65.0,
    'potato':       60.0,
    'tomato':       55.0,
    'wheat':        45.0,
    'maize':        50.0,
    'cotton':       45.0,
    'barley':       40.0,
    'faba bean':    45.0,
    'lentil':       40.0,
    'onion':        50.0,
    'garlic':       50.0,
    'mango':        45.0,
    'citrus':       50.0,
    'grapes':       40.0,
    'groundnut':    45.0,
    'soybean':      50.0,
    'chickpea':     35.0,
    'watermelon':   45.0,
    'coffee':       60.0,
    'sunflower':    40.0,
}
DEFAULT_MOIST_FALLBACK = 45.0

print(' Lookup tables loaded')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Smart fill function — operates on df_aug
# ══════════════════════════════════════════════════════════════

def smart_fill_nulls(df):
    df = df.copy()

    print(' NULL before filling:')
    cols_to_check = ['fertilizer','rainfall','moisture']
    for col in cols_to_check:
        n = df[col].isna().sum()
        pct = n / len(df) * 100
        print(f'  {col:<15}: {n:,} null ({pct:.1f}%)')

    # ────────────────────────────────────────────
    # 1. FERTILIZER — fill with default by crop
    # ────────────────────────────────────────────
    def fill_fert(row):
        if pd.notna(row['fertilizer']) and str(row['fertilizer']) not in ['nan','none','']:
            return str(row['fertilizer']).strip()
        crop = str(row.get('crop','')).lower().strip()
        # find closest match in lookup
        for key in DEFAULT_FERTILIZER:
            if key in crop or crop in key:
                return DEFAULT_FERTILIZER[key]
        return DEFAULT_FERT_FALLBACK

    mask_fert = df['fertilizer'].isna() | df['fertilizer'].isin(['nan','none',''])
    df.loc[mask_fert, 'fertilizer'] = df[mask_fert].apply(fill_fert, axis=1)

    # ────────────────────────────────────────────
    # 2. RAINFALL — fill with default by crop
    #    then add ±20% noise for variety
    # ────────────────────────────────────────────
    def fill_rain(row):
        if pd.notna(row['rainfall']) and float(row['rainfall']) > 0:
            return row['rainfall']
        crop = str(row.get('crop','')).lower().strip()
        base = DEFAULT_RAIN_FALLBACK
        for key in DEFAULT_RAINFALL:
            if key in crop or crop in key:
                base = DEFAULT_RAINFALL[key]
                break
        # add ±20% noise so rows aren't identical
        noise = np.random.uniform(-0.2, 0.2) * base
        return round(max(5.0, base + noise), 1)

    mask_rain = df['rainfall'].isna() | (df['rainfall'] <= 0)
    df.loc[mask_rain, 'rainfall'] = df[mask_rain].apply(fill_rain, axis=1)

    # ────────────────────────────────────────────
    # 3. MOISTURE — fill with default by crop
    #    then add ±15% noise
    # ────────────────────────────────────────────
    def fill_moist(row):
        if pd.notna(row['moisture']) and float(row['moisture']) > 0:
            return row['moisture']
        crop = str(row.get('crop','')).lower().strip()
        base = DEFAULT_MOIST_FALLBACK
        for key in DEFAULT_MOISTURE:
            if key in crop or crop in key:
                base = DEFAULT_MOISTURE[key]
                break
        noise = np.random.uniform(-0.15, 0.15) * base
        return round(min(95.0, max(10.0, base + noise)), 1)

    mask_moist = df['moisture'].isna() | (df['moisture'] <= 0)
    df.loc[mask_moist, 'moisture'] = df[mask_moist].apply(fill_moist, axis=1)

    # ────────────────────────────────────────────
    # 4. Final check
    # ────────────────────────────────────────────
    print('\n NULL after filling:')
    for col in cols_to_check:
        n = df[col].isna().sum()
        print(f'  {col:<15}: {n} null')

    print(f'\n Total rows: {len(df):,} (no rows dropped)')
    return df


# Apply to df_aug
df_aug = smart_fill_nulls(df_aug)

# Save updated version
df_aug.to_csv('./output/merged_dataset.csv', index=False)
print('\n merged_dataset.csv updated')

In [ ]:
# Check random sample after filling
print('Sample data after filling:')
sample = df_aug[['crop','nitrogen','phosphorus','potassium',
                  'ph','fertilizer','rainfall','moisture',
                  'soil_type','source']].sample(5, random_state=1)
print(sample.to_string(index=False))

print('\n NULL summary across all columns after filling:')
null_summary = df_aug.isnull().sum()
null_summary = null_summary[null_summary > 0]
if len(null_summary) == 0:
    print('   No NULL values in core columns!')
else:
    print(null_summary)

##  STEP 15 — Download Files

In [ ]:
!zip -r soil_data_v4_final.zip ./output/
files.download('soil_data_v4_final.zip')
print(' Download started!')

---
##  V4 Summary

### Applied Tiered Cleaning:
```
Full data (before)       →  all sources with NULLs
After dropping no-NPK    →  preserving maximum data
After dropping no-crop   →  crop must be known
After removing outliers  →  only 0.5% from each end
pH and soil_type         →  not dropped — filled from MALR ideal
Augmentation             →  ×3 with 5% noise
```

### Output Files:
```
train.jsonl              → 85% for training
val.jsonl                → 7%  for validation
test.jsonl               → 8%  for testing
merged_dataset.csv       → merged dataset
egypt_malr_ideal.json    → MALR Egypt lookup
```

### Next Step
Use `train.jsonl` in the **Fine-tuning on Mistral-7B + QLoRA** notebook